# Dim Part — Gold Layer

Enriched part dimension joining `foundation_parts` with `foundation_part_categories` and deriving production span metrics.

## What this notebook does

1. **Read** — Loads parts and part categories from silver.
2. **Transform** — Joins with part categories, derives `production_span_years` and `is_active` flag.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/gold/delta_volume/dim_part`.
4. **Register** — Creates the catalog table `lego_brickbase.gold.dim_part`.

## Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit


SILVER_PARTS      = "lego_brickbase.silver.foundation_parts"
SILVER_CATEGORIES = "lego_brickbase.silver.foundation_part_categories"
DELTA_TABLE_PATH  = "/Volumes/lego_brickbase/gold/delta_volume/dim_part"
CATALOG_TABLE     = "lego_brickbase.gold.dim_part"

## Read and Transform

Join parts with part categories. Derive `production_span_years` (`year_to - year_from`) and `is_active` (true when `year_to` is null, meaning the part is still in production).

In [ ]:
df_parts = spark.table(SILVER_PARTS)
df_categories = spark.table(SILVER_CATEGORIES)

df_dim = (
    df_parts
    .join(
        df_categories,
        df_parts["part_category_key"] == df_categories["part_category_key"],
        "left",
    )
    .select(
        df_parts["part_key"],
        df_parts["part_number"],
        df_parts["part_name"],
        df_categories["part_category_name"],
        df_parts["year_from"],
        df_parts["year_to"],
        (col("year_to") - col("year_from")).alias("production_span_years"),
        when(col("year_to").isNull(), lit(True)).otherwise(lit(False)).alias("is_active"),
        df_parts["part_url"],
        df_parts["part_image_url"],
        df_parts["external_ids"],
    )
)

df_dim.printSchema()
df_dim.display(10, truncate=False)

## Write to Delta Volume, Register Catalog Table, and Apply Constraints

In [ ]:
(
    df_dim
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE} (
        part_key              STRING  NOT NULL COMMENT 'Surrogate key for the part dimension',
        part_number           STRING           COMMENT 'Official LEGO part number as assigned in the Rebrickable catalog',
        part_name             STRING           COMMENT 'Descriptive name of the LEGO part',
        part_category_name    STRING           COMMENT 'Category name the part belongs to (e.g. Brick, Technic, Minifig)',
        year_from             INTEGER          COMMENT 'First year this part appeared in a LEGO set',
        year_to               INTEGER          COMMENT 'Last year this part appeared in a LEGO set; NULL if the part is still in production',
        production_span_years INTEGER          COMMENT 'Number of years the part has been in production (year_to - year_from); NULL if still active',
        is_active             BOOLEAN          COMMENT 'True if the part is still in active production (year_to is NULL)',
        part_url              STRING           COMMENT 'URL to the part detail page on Rebrickable',
        part_image_url        STRING           COMMENT 'URL to the canonical image of the part',
        external_ids          STRING           COMMENT 'JSON-encoded map of external part identifiers from other catalog systems (e.g. BrickLink, LDraw)',
        CONSTRAINT dim_part_pk PRIMARY KEY (part_key)
    )
    COMMENT 'Part dimension joining silver foundation_parts with part categories. Includes derived production_span_years and is_active flag.'
""")

spark.sql(f"""
    INSERT INTO {CATALOG_TABLE}
    SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")

print(f"Catalog table ready with constraints: {CATALOG_TABLE}")